# Mouse embryo — CellSTIC tutorial
**Dataset:** Mouse embryo Stereo-seq; per-stage input under `data/mouse_embryo/<stage>/raw/`; ligand–receptor map `data/mouse_embryo/LR.csv`. See `data/mouse_embryo/README.md`.


## 1. Repository path


In [ ]:
from pathlib import Path
import sys
import torch
import scanpy as sc

# Add project root to path
try:
    project_root = Path(__file__).resolve().parent.parent
except NameError:
    _cwd = Path.cwd()
    project_root = _cwd if (_cwd / "model").is_dir() else _cwd.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root.resolve()}")

## 2. Imports


In [ ]:
from utils.tools.seed_utils import set_global_seed
from typing import List

from utils.loader import load_mouse_embryo
from utils.train import load_config
from model.cellstic import CellSTIC
from pipeline.trainer import CellSTICTrainer
from pipeline.evaluator import CellSTICEvaluator
from pipeline.analyzer import TimeSequenceAnalysis
from pipeline.analyzer import SingleLevelAnalysis
from utils import ModelUtils
from utils.metrics import SpatialVisualizer

ANNOTATION_MODEL_MAP = {"Brain": "mouse_brain", "Liver": "mouse_liver"}

set_global_seed()  # once before train/eval; pipeline does not set RNG here


## 3. Configuration

Edit this section before running the notebook. The switches below keep the notebook easy to reuse while preserving the original imperative workflow.


In [ ]:
# --- Single-stage run configuration ---
STAGE = "14.5"
ORGAN = "Brain"  # "Brain" or "Liver"
RUN_TRAIN = True

# --- Optional visualization switch ---
GENERATE_ANNOTATION_SPATIAL_MAP = False

# --- Optional time-series analysis switch ---
RUN_TIME_SERIES_ANALYSIS = False
TIME_SERIES_STAGES = ["12.5", "14.5"]

# --- Common paths ---
work_dir = project_root / "data" / "mouse_embryo" / STAGE
raw_path = work_dir / "raw"
config_path = work_dir / "config" / "mouse_embryos_config.yaml"
lr_path = (work_dir.parent / "LR.csv").resolve()
output_path = work_dir / "output" / ORGAN
model_path = work_dir / "model" / ORGAN
preprocess_path = work_dir / "preprocess" / ORGAN

for _p in [raw_path, output_path, model_path, preprocess_path]:
    _p.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Stage: {STAGE}")
print(f"Organ: {ORGAN}")
print(f"Train model: {RUN_TRAIN}")
print(f"Device: {device}")
print(f"Raw path: {raw_path}")
print(f"Config path: {config_path}")
print(f"LR path: {lr_path}")
print(f"Model path: {model_path}")
print(f"Output path: {output_path}")


## 4. Optional annotation spatial map


In [ ]:
if GENERATE_ANNOTATION_SPATIAL_MAP:
    raw_files_vis = list(raw_path.glob("*.h5ad"))
    if not raw_files_vis:
        raise FileNotFoundError(
            f"No raw h5ad found under raw_path: {raw_path.absolute()}"
        )

    rna_adata_vis = sc.read_h5ad(raw_files_vis[0])
    if "annotation" not in rna_adata_vis.obs:
        raise ValueError("`annotation` column not found in adata.obs; cannot colour spatial map.")

    rna_adata_vis = rna_adata_vis.copy()
    rna_adata_vis.obs["cluster"] = rna_adata_vis.obs["annotation"].astype(str).astype("category")

    annotation_spatial_dir = work_dir / "output" / "annotation_spatial"
    annotation_spatial_dir.mkdir(parents=True, exist_ok=True)
    save_path_vis = annotation_spatial_dir / "annotation_spatial.svg"

    SpatialVisualizer.generate_spatial_domain_visualization(
        adata_source=rna_adata_vis,
        save_path=str(save_path_vis),
    )

    print(f"Annotation spatial map saved to: {save_path_vis}")
else:
    print("Skip annotation spatial map generation.")

## 5. Load configuration and data

Load YAML config and `AnnData` + ligand–receptor map for the selected stage and organ.


In [ ]:
config = load_config(config_path)

rna_adata, lr = load_mouse_embryo(
    raw_path,
    preprocess_path,
    use_cache=True,
    annotation_value=ORGAN,
    lr_path=lr_path,
    annotation_model_map=ANNOTATION_MODEL_MAP,
)


## 6. Train or load model

Train `CellSTIC` when `RUN_TRAIN` is True; otherwise load weights from `model_path`.


In [ ]:
if RUN_TRAIN:
    model = CellSTIC(config.model, device)
    CellSTICTrainer(
        model,
        config,
        model_path=model_path,
        device=device,
        ligand_receptor_map=lr,
    ).train(
        modality_datas=[rna_adata],
        is_train_ccc=True,
        is_train_feature=True,
    )
else:
    model = ModelUtils.load_model(
        model_path=model_path,
        config=config,
        device=device,
    )


## 7. Evaluate features and CCC

Fused-feature metrics, then ligand–receptor edge probabilities on the spatial graph.


In [ ]:
evaluator = CellSTICEvaluator(
    model,
    config,
    ligand_receptor_map=lr,
    model_path=model_path,
    output_path=output_path,
    device=device,
)

evaluator.evaluate_mutiple_feature(
    modality_datas=[rna_adata],
    auto_n_clusters=7,
)

pos_edge_probs_np, edge_type_map, m0 = evaluator.evaluate_ccc_precision(
    modality_datas=[rna_adata],
)


## 8. Single-level analysis

Cell-type × LR heatmaps on the evaluated CCC output (`m0`).


In [ ]:
if ORGAN == "Brain":
    cell_type_filter = [
        "Blood",
        "Immune",
        "Gastrulation",
        "Vascular",
        "Neuron",
        "Ectoderm",
        "Fibroblast",
        "Neuroblast:",
        "Glioblast",
        "Radial glia",
    ]
else:
    cell_type_filter = None

single_level_analysis = SingleLevelAnalysis(
    adata=m0,
    pos_edge_probs_np=pos_edge_probs_np,
    edge_type_map=edge_type_map,
    output_path=output_path,
    cell_type_key="cell_type",
    threshold=0.5,
    lr_filter=["F2:F2r", "Lpar3:Adgre5", "Nts:Sort1", "Plg:Pard3", "Thbs4:Cd36"],
    annotation_filter=["Brain", "Liver"],
    cell_type_filter=cell_type_filter,
)
single_level_analysis.run_cell_type_heatmaps()

print("Single-stage workflow completed.")
print(f"Results saved under: {output_path}")


## 9. Optional cross-stage time-series analysis


In [ ]:
if RUN_TIME_SERIES_ANALYSIS:
    # Cross-stage CCI summaries (uses per-stage folders under data/mouse_embryo/)
    work_dir_cci = project_root / "data" / "mouse_embryo"
    output_path_cci = work_dir_cci / "output"
    raw_path_cci = work_dir_cci / "raw"
    adata_path_cci = project_root / "data" / "mouse_embryo"
    lr_path_cci = (work_dir_cci.parent / "LR.csv").resolve()  # reserved for LR-aware plots if extended

    output_path_cci.mkdir(parents=True, exist_ok=True)
    raw_path_cci.mkdir(parents=True, exist_ok=True)

    # Organs and LR identifiers expected by TimeSequenceAnalysis (hyphenated ligand-receptor names)
    annotation_filter = ["Brain", "Liver"]
    lr_filter = ["F2-F2r", "Lpar3-Adgre5", "Nts-Sort1", "Plg-Pard3", "Thbs4-Cd36"]
    region_lr_map = {"Brain": "Nts-Sort1", "Liver": "F2-F2r"}

    # Multi-stage runner over TIME_SERIES_STAGES
    time_sequence_analysis = TimeSequenceAnalysis(
        stages=TIME_SERIES_STAGES,
        raw_path=raw_path_cci,
        output_path=output_path_cci,
        annotation_filter=annotation_filter,
        lr_filter=lr_filter,
        spatial_root=str(adata_path_cci),
        fig_format="svg",
    )

    # Cell counts per stage (bar / line export)
    time_sequence_analysis.count_cell_number(recompute=True, font_size=12)
    # CCC edge counts per stage
    time_sequence_analysis.count_edge_number(recompute=True, font_size=12)
    # Precision / recall style efficiency curves across stages
    time_sequence_analysis.plot_efficiency_metrics(
        recompute=True,
        region_lr_map=region_lr_map,
        font_size=8,
    )
    # Mean LR strength bars by cell type (per region key in region_lr_map)
    time_sequence_analysis.plot_celltype_strength_bars(
        recompute=True,
        region_lr_map=region_lr_map,
        font_size=8,
    )
    # Graph density (edges per cell) trajectories across stages
    time_sequence_analysis.plot_graph_density_over_stages(
        recompute=True,
        font_size=12,
    )
    # Strongest communicating nodes over developmental stages
    time_sequence_analysis.plot_strong_nodes_over_stages(
        recompute=True,
        region_lr_map=region_lr_map,
        font_size=8,
    )
    # LR strength vs spatial distance across stages
    time_sequence_analysis.plot_strength_vs_distance_over_stages(
        recompute=True,
        region_lr_map=region_lr_map,
        font_size=12,
    )
    # CCDF of degree-weighted edge strength (per region LR)
    time_sequence_analysis.plot_ccdf_degree_strength(
        recompute=True,
        region_lr_map=region_lr_map,
        font_size=8,
    )

    print(f"Time-series analysis completed for stages: {TIME_SERIES_STAGES}")
    print(f"Time-series outputs saved under: {output_path_cci}")
else:
    print("Skip time-series analysis.")